# Lab 03 - Delta Lake Deep Dive
## Insurance Claims Analytics

---

### Learning Objectives

By the end of this lab, you will be able to:

1. **Explain** why Delta Lake exists and what problems it solves over plain Parquet
2. **Use ACID transactions** (MERGE) to safely upsert data without corruption
3. **Leverage schema enforcement** to prevent bad data and **schema evolution** to adapt to changes
4. **Time travel** to query, restore, and audit historical versions of your data
5. **Inspect the Delta Log** to understand how Delta Lake tracks every change
6. **Add CHECK constraints** to enforce data quality rules at the table level

---
## Step 0: Generate Data

> Run this cell to create claims data in Hive Metastore. No Unity Catalog needed.

In [ ]:
# ============================================================
# STEP 0: Generate claims data
# Run this ONCE before starting exercises
# ============================================================

import random
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

# Derive a unique database name per participant
_user = spark.sql("SELECT current_user()").first()[0]
_user_clean = _user.split("@")[0].replace(".", "_").replace("-", "_").lower()
USER_DB = f"training_lab_{_user_clean}"
print(f"Your personal database: {USER_DB}")

spark.sql(f"CREATE DATABASE IF NOT EXISTS {USER_DB}")

# Clean up tables from previous runs for idempotency
for t in ["claims_demo", "claims_raw", "delta_exploration_tbl"]:
    spark.sql(f"DROP TABLE IF EXISTS {USER_DB}.{t}")


# Generate 1000 insurance claims
claim_types = ["PROPERTY", "LIABILITY", "AUTO", "HEALTH", "MARINE", "AVIATION"]
claim_statuses = ["OPEN", "CLOSED", "PENDING", "UNDER_REVIEW"]
entity_ids = [f"ENT_{str(i).zfill(3)}" for i in range(1, 19)]

claims_data = []
for i in range(1, 1001):
    claims_data.append((
        f"CLM-{str(i).zfill(6)}",
        f"POL-{str(random.randint(1, 500)).zfill(6)}",
        random.choice(entity_ids),
        f"2024-{random.randint(1,12):02d}-{random.randint(1,28):02d}",
        round(random.uniform(1000, 500000), 2),
        random.choice(claim_types),
        random.choice(claim_statuses),
        f"ADJ-{str(random.randint(1, 50)).zfill(3)}"
    ))

claims_schema = StructType([
    StructField("claim_id", StringType()), StructField("policy_id", StringType()),
    StructField("axa_entity_id", StringType()), StructField("claim_date", StringType()),
    StructField("claim_amount", DoubleType()), StructField("claim_type", StringType()),
    StructField("claim_status", StringType()), StructField("adjuster_id", StringType())
])

claims_df = spark.createDataFrame(claims_data, claims_schema)
claims_df.write.mode("overwrite").format("delta").saveAsTable(f"{USER_DB}.claims_raw")

print(f"Generated {claims_df.count()} claims in {USER_DB}.claims_raw")


---
## Setup

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, lit, when, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, DateType, IntegerType

# ---- Configuration ----
DATABASE = USER_DB
DEMO_TABLE = f"{DATABASE}.claims_demo"
DELTA_EXPLORATION_PATH = f"{DATABASE}.delta_exploration_tbl"

print(f"Database:    {DATABASE}")
print(f"Demo Table:  {DEMO_TABLE}")


---
## Section 1: What Makes Delta Lake Special?

### WHY Delta Lake?

Imagine a data lake built on plain Parquet files. It works... until it doesn't:
- An ETL job crashes mid-write: **half your data is corrupted**
- Two jobs write to the same folder simultaneously: **data is inconsistent**
- Someone changes the schema upstream: **readers break silently**
- You need last week's version of the data: **too late, it's been overwritten**

Delta Lake solves ALL of these problems by adding a **transaction log** on top of Parquet files.

In [1]:
displayHTML("""
<div style="display: flex; gap: 24px; flex-wrap: wrap; margin: 20px 0; font-family: 'Cascadia Code', Consolas, monospace; font-size: 13px;">
  <div style="flex: 1; min-width: 220px; background: #1B3A4B; color: #F9F7F4; border-radius: 10px; padding: 18px; line-height: 1.8;">
    <div style="color: #FF6F47; font-weight: 700; font-size: 14px; margin-bottom: 10px; font-family: 'Segoe UI', sans-serif;">PARQUET alone</div>
    <span style="color: #9E9E9E;">data/</span><br>
    &nbsp;&nbsp;+-- part-001.parquet<br>
    &nbsp;&nbsp;+-- part-002.parquet<br>
    &nbsp;&nbsp;+-- part-003.parquet<br><br>
    <span style="color: #FF6F47;">Just files.</span><br>
    <span style="color: #FF6F47;">No guarantees.</span>
  </div>
  <div style="flex: 1; min-width: 280px; background: #00A972; color: #FFFFFF; border-radius: 10px; padding: 18px; line-height: 1.8;">
    <div style="color: #FFFDE7; font-weight: 700; font-size: 14px; margin-bottom: 10px; font-family: 'Segoe UI', sans-serif;">DELTA LAKE</div>
    <span style="color: #C8E6C9;">data/</span><br>
    &nbsp;&nbsp;+-- part-001.parquet<br>
    &nbsp;&nbsp;+-- part-002.parquet<br>
    &nbsp;&nbsp;+-- part-003.parquet<br>
    &nbsp;&nbsp;+-- <span style="color: #FFFDE7; font-weight: bold;">_delta_log/</span><br>
    &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;+-- 00000.json<br>
    &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;+-- 00001.json<br>
    &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;+-- 00002.json<br><br>
    <span style="color: #FFFDE7;">Files + Transaction Log</span><br>
    <span style="color: #FFFDE7;">= Database-like reliability</span>
  </div>
</div>
""")

PARQUET alone 
 data/ 
   +-- part-001.parquet 
   +-- part-002.parquet 
   +-- part-003.parquet 
 Just files. 
 No guarantees. 
 
 
 DELTA LAKE 
 data/ 
   +-- part-001.parquet 
   +-- part-002.parquet 
   +-- part-003.parquet 
   +-- _delta_log/ 
       +-- 00000.json 
       +-- 00001.json 
       +-- 00002.json 
 Files + Transaction Log 
 = Database-like reliability

The <code>_delta_log</code> is the secret sauce. Every write, update, delete, and schema change is recorded as an atomic JSON entry.

### Demo 1.1 - Read Claims Data

First, let's read our insurance claims raw data and inspect its schema.

In [ ]:
# Demo 1.1 - Read the raw claims data
claims_raw = spark.read.table(f"{USER_DB}.claims_raw")

print(f"Claims loaded: {claims_raw.count()} rows")
claims_raw.printSchema()
display(claims_raw.limit(5))

### Exercise 1.2 - Write as Delta Table

Now let's save this data as a Delta table -- going from "just Parquet files" to a full transactional table.

**Fill in the blanks** to write as Delta format with overwrite mode.

In [ ]:
# Exercise 1.2 - Write claims as a Delta table

(
    claims_raw
    .write
    .format("________")
    .mode("________")
    .saveAsTable(DEMO_TABLE)
)

print(f"Delta table created: {DEMO_TABLE}")

In [ ]:
# Verification 1.2
table_count = spark.table(DEMO_TABLE).count()
assert table_count > 0, "Table is empty - check format and mode"
detail = spark.sql(f"DESCRIBE DETAIL {DEMO_TABLE}").select("format").first()[0]
assert detail == "delta", f"Expected delta format, got {detail}"
print(f"Exercise 1.2 passed! Delta table created with {table_count} rows")

In [1]:
displayHTML("""
<details>
<summary>Click to reveal more hints</summary>

**Hints:**
- The first blank is the storage format — Delta Lake's format name is simply its own name, lowercase
- The second blank is the write mode — since this is our first load and we want to replace any existing data, think: "write over"
</details>
""")

Click to reveal more hints 

**Hints:**
- The first blank is the storage format — Delta Lake's format name is simply its own name, lowercase
- The second blank is the write mode — since this is our first load and we want to replace any existing data, think: "write over"

### Exercise 1.3 - Inspect Delta Metadata

Use `DESCRIBE DETAIL` to see the metadata that Delta Lake maintains: format, location, number of files, size, and more.

In [ ]:
# Exercise 1.3 - Inspect Delta table metadata
detail_df = spark.sql(f"________ {DEMO_TABLE}")
display(detail_df.select("format", "location", "numFiles", "sizeInBytes", "properties"))

In [1]:
displayHTML("""
<details style="margin: 10px 0;">
  <summary style="cursor: pointer; color: #1B3A4B; font-weight: bold; padding: 8px 0;">Click to reveal hints</summary>
  <div style="background: #F9F7F4; padding: 12px 16px; border-radius: 8px; margin-top: 8px; border-left: 4px solid #F9A825;">
    <ul style="margin: 8px 0; padding-left: 20px;"><li style="margin: 6px 0; color: #1B1F24;">Answer: <code>DESCRIBE DETAIL</code></li></ul>
  </div>
</details>
""")

Click to reveal hints 
 
 Answer: DESCRIBE DETAIL

In [1]:
displayHTML("""
<div style="border-left: 4px solid #00A972; padding: 10px 15px; margin: 10px 0; background-color: #F1F8E9; color: #1B1F24;">
<strong>Key Takeaway:</strong> Delta Lake = Parquet files + a transaction log (<code>_delta_log</code>). This simple addition gives you ACID transactions, schema enforcement, time travel, and audit history. Writing a Delta table is as simple as <code>.format("delta")</code>.
</div>
""")

Key Takeaway: Delta Lake = Parquet files + a transaction log ( _delta_log ). This simple addition gives you ACID transactions, schema enforcement, time travel, and audit history. Writing a Delta table is as simple as .format("delta") .

---
## Section 2: ACID Transactions

### WHY ACID?

Consider this real scenario: Your nightly ETL processes 50,000 new insurance claims. Halfway through, the cluster crashes.

- **Without ACID (plain Parquet):** 25,000 records are written. Your table now has partial, inconsistent data. Downstream dashboards show wrong numbers. You must manually identify and clean up the mess.
- **With ACID (Delta Lake):** The entire write either succeeds or fails completely. If it crashes, the table stays exactly as it was before. Zero cleanup needed.

ACID stands for:
- **A**tomic: All or nothing -- no partial writes
- **C**onsistent: Data always satisfies defined rules
- **I**solated: Concurrent operations don't interfere
- **D**urable: Committed data is never lost

The most powerful ACID operation is **MERGE** (also called UPSERT): UPDATE existing rows + INSERT new rows, all in a single atomic transaction.

### Exercise 2.1 - MERGE (Upsert) Demo

We receive a batch of updated claims: some existing claims have new statuses, and there are also brand-new claims. MERGE handles both in one operation.

In [ ]:
# Setup: Record current count and create updates DataFrame
before_count = spark.table(DEMO_TABLE).count()
print(f"Records BEFORE merge: {before_count}")

# Get 3 existing claim IDs to update + 2 brand new ones
existing_ids = [row.claim_id for row in spark.table(DEMO_TABLE).select("claim_id").limit(3).collect()]

updates_data = []
for cid in existing_ids:
    updates_data.append((cid, None, None, None, None, None, "CLOSED", None))
updates_data.append(("CLM-NEW-001", "POL-999", "ENT_001", "2025-01-15", 42000.0, "AUTO", "OPEN", "ADJ-010"))
updates_data.append(("CLM-NEW-002", "POL-888", "ENT_002", "2025-01-16", 15500.0, "PROPERTY", "OPEN", "ADJ-011"))

updates_schema = StructType([
    StructField("claim_id", StringType()),
    StructField("policy_id", StringType()),
    StructField("axa_entity_id", StringType()),
    StructField("claim_date", StringType()),
    StructField("claim_amount", DoubleType()),
    StructField("claim_type", StringType()),
    StructField("claim_status", StringType()),
    StructField("adjuster_id", StringType())
])

updates_df = spark.createDataFrame(updates_data, updates_schema)
updates_df.createOrReplaceTempView("claims_updates")
print(f"Updates: {updates_df.count()} rows ({len(existing_ids)} updates + 2 new)")
display(updates_df)


In [ ]:
# Exercise 2.1: Execute MERGE
# Single atomic transaction: ALL changes apply or NONE do
spark.sql(f"""
MERGE INTO {DEMO_TABLE} AS target
USING claims_updates AS source
ON target.claim_id = source.claim_id
WHEN MATCHED THEN
  ________ target.claim_status = source.claim_status
WHEN NOT MATCHED THEN
  INSERT *
""")


In [1]:
displayHTML("""
<details style="margin: 10px 0;">
  <summary style="cursor: pointer; color: #1B3A4B; font-weight: bold; padding: 8px 0;">Click to reveal hints</summary>
  <div style="background: #F9F7F4; padding: 12px 16px; border-radius: 8px; margin-top: 8px; border-left: 4px solid #F9A825;">
    <ul style="margin: 8px 0; padding-left: 20px;"><li style="margin: 6px 0; color: #1B1F24;">Answer: <code>UPDATE SET</code></li></ul>
  </div>
</details>
""")

Click to reveal hints 
 
 Answer: UPDATE SET

In [ ]:
# Exercise 2.2 - Verify counts and results
after_count = spark.table(DEMO_TABLE).count()
print(f"Records BEFORE merge: {before_count}")
print(f"Records AFTER merge:  {after_count}")
print(f"New records added:    {after_count - before_count}")

print("\nUpdated records (should show CLOSED status):")
spark.table(DEMO_TABLE).filter(col("claim_id").isin(existing_ids)).select("claim_id", "claim_status")

print("New records:")
spark.table(DEMO_TABLE).filter(col("claim_id").startswith("CLM-NEW"))

In [1]:
displayHTML("""
<div style="border-left: 4px solid #00A972; padding: 10px 15px; margin: 10px 0; background-color: #F1F8E9; color: #1B1F24;">
<strong>Key Takeaway:</strong> MERGE is the workhorse of Delta Lake. It atomically handles updates and inserts in one statement. If anything fails, the entire operation rolls back. This is impossible with plain Parquet.
</div>
""")

Key Takeaway: MERGE is the workhorse of Delta Lake. It atomically handles updates and inserts in one statement. If anything fails, the entire operation rolls back. This is impossible with plain Parquet.

---
## Section 3: Schema Enforcement & Evolution

### WHY Schema Enforcement?

In production, data arrives from many sources. Without schema enforcement:
- An upstream system adds a field: your table silently gains an unexpected column
- A type changes from `double` to `string`: your analytics queries break
- A column is renamed: nulls appear everywhere

Delta Lake enforces schema on every write. If incoming data doesn't match, the write **fails immediately** rather than silently corrupting your data.

But sometimes you **want** to evolve the schema (e.g., adding a new column). Delta supports that too, with an explicit opt-in: `mergeSchema`.

### Demo 3.1 - Schema Enforcement (Rejected Write)

Let's try to append data with the wrong schema. Delta Lake should **reject** this.

In [ ]:
# Demo 3.1 - Try appending data with wrong schema
wrong_schema = StructType([
    StructField("id", StringType()),
    StructField("amount", IntegerType()),
    StructField("wrong_column", StringType())
])

wrong_schema_df = spark.createDataFrame([
    ("X1", 100, "bad_field"),
    ("X2", 200, "bad_field")
], wrong_schema)

try:
    wrong_schema_df.write.format("delta").mode("append").saveAsTable(DEMO_TABLE)
    print("Write succeeded -- this should NOT happen!")
except Exception as e:
    print("Schema enforcement BLOCKED the write (as expected).")
    print(f"Error: {str(e)[:200]}")


### Exercise 3.2 - Schema Evolution (Adding a Column)

Now let's intentionally evolve the schema by adding a new `priority` column. We use the `mergeSchema` option to explicitly allow this.

**Fill in the blank** with the correct option name.

In [ ]:
# Exercise 3.2 - Schema evolution: add a new column
evolved_schema = StructType([
    StructField("claim_id", StringType()),
    StructField("policy_id", StringType()),
    StructField("axa_entity_id", StringType()),
    StructField("claim_date", StringType()),
    StructField("claim_amount", DoubleType()),
    StructField("claim_type", StringType()),
    StructField("claim_status", StringType()),
    StructField("adjuster_id", StringType()),
    StructField("priority", StringType())
])

evolved_df = spark.createDataFrame([
    ("CLM-EVOLVED-001", "POL-777", "ENT_001", "2025-02-01", 8000.0, "LIABILITY", "OPEN", "ADJ-012", "HIGH")
], evolved_schema)

(
    evolved_df
    .write
    .format("delta")
    .mode("append")
    .option("________", "true")
    .saveAsTable(DEMO_TABLE)
)

print("Schema evolution succeeded! New schema:")
spark.table(DEMO_TABLE).printSchema()


In [ ]:
# Verification 3.2
schema_cols = [f.name for f in spark.table(DEMO_TABLE).schema.fields]
assert "priority" in schema_cols, "Column 'priority' not found - check the mergeSchema option"
print(f"Exercise 3.2 passed! Schema evolved - columns: {schema_cols}")

In [1]:
displayHTML("""
<details>
<summary>Click to reveal more hints</summary>

**Hints:**
- You need an option that tells Delta Lake to accept new columns in incoming data
- The option name combines two words: "merge" + "Schema"
- It's a camelCase string, set to `"true"`
</details>
""")

Click to reveal more hints 

**Hints:**
- You need an option that tells Delta Lake to accept new columns in incoming data
- The option name combines two words: "merge" + "Schema"
- It's a camelCase string, set to `"true"`

In [1]:
displayHTML("""
<div style="border-left: 4px solid #00A972; padding: 10px 15px; margin: 10px 0; background-color: #F1F8E9; color: #1B1F24;">
<strong>Key Takeaway:</strong> Schema enforcement is your safety net -- it blocks writes that don't match the expected schema. Schema evolution (<code>mergeSchema</code>) lets you explicitly opt in to adding new columns. You get both protection and flexibility.
</div>
""")

Key Takeaway: Schema enforcement is your safety net -- it blocks writes that don't match the expected schema. Schema evolution ( mergeSchema ) lets you explicitly opt in to adding new columns. You get both protection and flexibility.

---
## Section 4: Time Travel

### WHY Time Travel?

Picture this: It's 3 AM. A junior engineer runs a DELETE query without a WHERE clause. Every single claim record is gone.

**Without Delta Lake:** You call the backup team, restore from the last nightly backup, and lose an entire day of data. The incident takes 6 hours to resolve.

**With Delta Lake:** You run `RESTORE TABLE ... TO VERSION AS OF n` and the table is back in 30 seconds. Crisis averted.

Every operation on a Delta table creates a new **version**. You can:
- **Query** any historical version
- **Compare** versions to see what changed
- **Restore** to any previous version

In [ ]:
# Exercise 4.1 - View the complete history of operations
spark.sql(f"""
________ {DEMO_TABLE}
""")


In [1]:
displayHTML("""
<details style="margin: 10px 0;">
  <summary style="cursor: pointer; color: #1B3A4B; font-weight: bold; padding: 8px 0;">Click to reveal hints</summary>
  <div style="background: #F9F7F4; padding: 12px 16px; border-radius: 8px; margin-top: 8px; border-left: 4px solid #F9A825;">
    <ul style="margin: 8px 0; padding-left: 20px;"><li style="margin: 6px 0; color: #1B1F24;">Answer: <code>DESCRIBE HISTORY</code></li></ul>
  </div>
</details>
""")

Click to reveal hints 
 
 Answer: DESCRIBE HISTORY

### Demo 4.2 - Simulate a Disaster (DELETE without WHERE)

Let's simulate the 3 AM disaster. We'll record the count, then delete records.

In [ ]:
# Record current count before the disaster
count_before_delete = spark.table(DEMO_TABLE).count()
print(f"Records before disaster: {count_before_delete}")

In [ ]:
# Demo 4.2 - Simulate disaster: delete ALL claims with status 'OPEN'
spark.sql(f"""
DELETE FROM {DEMO_TABLE}
WHERE claim_status = 'OPEN'
""")


In [ ]:
count_after_delete = spark.table(DEMO_TABLE).count()
print(f"Records BEFORE delete: {count_before_delete}")
print(f"Records AFTER delete:  {count_after_delete}")
print(f"Records LOST:          {count_before_delete - count_after_delete}")

### Exercise 4.3 - Query a Previous Version

Before restoring, let's verify the data still exists in a previous version.

**Fill in the blank** to query the version before the delete.

In [ ]:
# Exercise 4.3 - Query a previous version
history_df = spark.sql(f"DESCRIBE HISTORY {DEMO_TABLE}")
delete_version = history_df.filter(col("operation") == "DELETE").select("version").first()[0]
version_before_delete = delete_version - 1
print(f"DELETE happened at version {delete_version}, querying version {version_before_delete}")

old_data = spark.sql(f"""
    SELECT COUNT(*) AS record_count
    FROM {DEMO_TABLE}
    ________  {version_before_delete}
""")old_data

In [ ]:
# Verification 4.3
old_count = old_data.first()[0]
assert old_count == count_before_delete, f"Expected {count_before_delete} but got {old_count} - check VERSION AS OF syntax"
print(f"Exercise 4.3 passed! Historical version has {old_count} records (matches pre-delete count)")

In [1]:
displayHTML("""
<details>
<summary>Click to reveal more hints</summary>

**Hints:**
- Delta Lake time travel uses a special SQL clause to query a specific version
- The syntax is three words: `VERSION` + `AS` + `OF` followed by the version number
- Place it right after the table name in the `FROM` clause
</details>
""")

Click to reveal more hints 

**Hints:**
- Delta Lake time travel uses a special SQL clause to query a specific version
- The syntax is three words: `VERSION` + `AS` + `OF` followed by the version number
- Place it right after the table name in the `FROM` clause

### Exercise 4.4 - RESTORE the Table

Now let's fix the disaster. **Fill in the blank** to restore to the correct version.

In [ ]:
# Exercise 4.4 - Restore the table

spark.sql(f"""
    RESTORE TABLE {DEMO_TABLE}
    TO ________  {version_before_delete}
""")print(f"Table restored to version {version_before_delete}!")

In [ ]:
# Verification 4.4
restored_count = spark.table(DEMO_TABLE).count()
assert restored_count == count_before_delete, f"Expected {count_before_delete} but got {restored_count} - check RESTORE syntax"
print(f"Exercise 4.4 passed! Table restored to {restored_count} records")

In [1]:
displayHTML("""
<details>
<summary>Click to reveal more hints</summary>

**Hints:**
- The `RESTORE TABLE` command uses the same time travel syntax you just used in Exercise 4.3
- After `TO`, use the same three-word clause: `VERSION AS OF` followed by the version number
</details>
""")

Click to reveal more hints 

**Hints:**
- The `RESTORE TABLE` command uses the same time travel syntax you just used in Exercise 4.3
- After `TO`, use the same three-word clause: `VERSION AS OF` followed by the version number

In [ ]:
# Exercise 4.5 - Verify the restore worked
count_after_restore = spark.table(DEMO_TABLE).count()

print(f"Records BEFORE delete:   {count_before_delete}")
print(f"Records AFTER delete:    {count_after_delete}")
print(f"Records AFTER restore:   {count_after_restore}")
print(f"\nRestoration successful: {count_after_restore == count_before_delete}")

In [1]:
displayHTML("""
<div style="border-left: 4px solid #00A972; padding: 10px 15px; margin: 10px 0; background-color: #F1F8E9; color: #1B1F24;">
<strong>Key Takeaway:</strong> Delta Lake time travel keeps every version of your data. Use <code>VERSION AS OF n</code> to query historical versions, and <code>RESTORE TABLE ... TO VERSION AS OF n</code> to undo mistakes. This turns a 6-hour disaster recovery into a 30-second fix.
</div>
""")

Key Takeaway: Delta Lake time travel keeps every version of your data. Use VERSION AS OF n to query historical versions, and RESTORE TABLE ... TO VERSION AS OF n to undo mistakes. This turns a 6-hour disaster recovery into a 30-second fix.

---
## Section 5: Delta Log Internals

### WHY Understand the Delta Log?

When something goes wrong with a Delta table, the transaction history is where you look to debug. Use `DESCRIBE HISTORY` to see every operation, who did it, and when. This helps you diagnose failed writes, understand what happened during a merge, and verify operations.

In [1]:
displayHTML("""
<div style="display: flex; gap: 24px; flex-wrap: wrap; margin: 20px 0; font-family: 'Cascadia Code', Consolas, monospace; font-size: 13px;">
  <div style="flex: 1; min-width: 320px; background: #1B3A4B; color: #F9F7F4; border-radius: 10px; padding: 18px; line-height: 1.8;">
    <div style="color: #FFFDE7; font-weight: 700; font-size: 14px; margin-bottom: 10px; font-family: 'Segoe UI', sans-serif;">_delta_log/ Structure</div>
    +-- <span style="color: #81C784;">00000000000000000000.json</span>&nbsp;&nbsp;<span style="color: #9E9E9E;">&larr; Version 0: CREATE TABLE</span><br>
    +-- <span style="color: #81C784;">00000000000000000001.json</span>&nbsp;&nbsp;<span style="color: #9E9E9E;">&larr; Version 1: INSERT</span><br>
    +-- <span style="color: #81C784;">00000000000000000002.json</span>&nbsp;&nbsp;<span style="color: #9E9E9E;">&larr; Version 2: UPDATE</span><br>
    +-- <span style="color: #9E9E9E;">...</span>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;<span style="color: #9E9E9E;">&larr; Each file = one atomic transaction</span>
  </div>
  <div style="flex: 1; min-width: 280px; background: #00A972; color: #FFFFFF; border-radius: 10px; padding: 18px; line-height: 1.8;">
    <div style="color: #FFFDE7; font-weight: 700; font-size: 14px; margin-bottom: 10px; font-family: 'Segoe UI', sans-serif;">Each JSON file contains</div>
    <span style="color: #FFFDE7;">commitInfo:</span>&nbsp;&nbsp;WHO did WHAT and WHEN<br>
    <span style="color: #FFFDE7;">add:</span>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;New Parquet files added<br>
    <span style="color: #FFFDE7;">remove:</span>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Old Parquet files marked for removal<br>
    <span style="color: #FFFDE7;">metaData:</span>&nbsp;&nbsp;&nbsp;&nbsp;Schema changes
  </div>
</div>
""")

_delta_log/ Structure 
 +-- 00000000000000000000.json    ← Version 0: CREATE TABLE 
 +-- 00000000000000000001.json    ← Version 1: INSERT 
 +-- 00000000000000000002.json    ← Version 2: UPDATE 
 +-- ...                                      ← Each file = one atomic transaction 
 
 
 Each JSON file contains 
 commitInfo:   WHO did WHAT and WHEN 
 add:          New Parquet files added 
 remove:       Old Parquet files marked for removal 
 metaData:     Schema changes

### Explore 5.1 - Create a Delta Table at a Known Path

We'll create a small Delta table and use `DESCRIBE HISTORY` to inspect its transaction log.

In [ ]:
# Explore 5.1 - Create a small Delta table to explore its history
spark.sql(f"DROP TABLE IF EXISTS {DELTA_EXPLORATION_PATH}")

sample_claims = spark.createDataFrame([
    ("CLM-A001", "POL-100", 25000.0, "AUTO", "OPEN"),
    ("CLM-A002", "POL-101", 15000.0, "PROPERTY", "OPEN"),
    ("CLM-A003", "POL-102", 5000.0, "LIABILITY", "CLOSED")
], ["claim_id", "policy_id", "claim_amount", "claim_type", "claim_status"])

sample_claims.write.format("delta").mode("overwrite").saveAsTable(DELTA_EXPLORATION_PATH)
print(f"Delta table created: {DELTA_EXPLORATION_PATH}")


In [ ]:
# Explore 5.2 - View the transaction history
# In managed tables, we use DESCRIBE HISTORY instead of browsing _delta_log files
print(f"Transaction history for: {DELTA_EXPLORATION_PATH}")
print("---")
display(spark.sql(f"DESCRIBE HISTORY {DELTA_EXPLORATION_PATH}"))


In [ ]:
# Explore 5.3 - Inspect table details and query specific versions
print("=== Table Details ===")
display(spark.sql(f"DESCRIBE DETAIL {DELTA_EXPLORATION_PATH}"))

print("\n=== Version 0 data (original 3 records) ===")
display(spark.sql(f"SELECT * FROM {DELTA_EXPLORATION_PATH} VERSION AS OF 0"))


In [ ]:
# Explore 5.4 - Append new data and observe the new version
new_claims = spark.createDataFrame([
    ("CLM-A004", "POL-103", 32000.0, "AUTO", "OPEN"),
], ["claim_id", "policy_id", "claim_amount", "claim_type", "claim_status"])

new_claims.write.format("delta").mode("append").saveAsTable(DELTA_EXPLORATION_PATH)

print("Transaction history after append (2 versions now):")
display(spark.sql(f"DESCRIBE HISTORY {DELTA_EXPLORATION_PATH}"))

v0 = spark.sql(f"SELECT COUNT(*) FROM {DELTA_EXPLORATION_PATH} VERSION AS OF 0").first()[0]
v1 = spark.sql(f"SELECT COUNT(*) FROM {DELTA_EXPLORATION_PATH} VERSION AS OF 1").first()[0]
print(f"\nVersion 0: {v0} records, Version 1: {v1} records")


In [1]:
displayHTML("""
<div style="border-left: 4px solid #00A972; padding: 10px 15px; margin: 10px 0; background-color: #F1F8E9; color: #1B1F24;">
<strong>Key Takeaway:</strong> The <code>_delta_log</code> is a directory of JSON files, one per version. Each entry records exactly which Parquet files were added or removed, plus metadata. This is how Delta achieves ACID: readers always see a consistent snapshot based on the log.
</div>
""")

Key Takeaway: The _delta_log is a directory of JSON files, one per version. Each entry records exactly which Parquet files were added or removed, plus metadata. This is how Delta achieves ACID: readers always see a consistent snapshot based on the log.

---
## Section 6: Data Quality Constraints

### WHY Constraints?

Schema enforcement catches structural problems (wrong columns, wrong types). But what about **semantic** problems?

- A claim with a **negative amount** passes schema checks (it's still a valid `DOUBLE`)
- A claim with status **"BANANA"** passes schema checks (it's still a valid `STRING`)

CHECK constraints are your **last line of defense**. They enforce business rules at the table level, rejecting any write that violates them.

In [ ]:
# Exercise 6.1 - Add constraints to enforce data quality rules

# Claim amounts must be positive
spark.sql(f"ALTER TABLE {DEMO_TABLE} ________ positive_amount CHECK (claim_amount > 0)")
print("Constraint added: positive_amount")

# Claim status must be one of the allowed values
spark.sql(f"ALTER TABLE {DEMO_TABLE} ________ valid_status CHECK (claim_status IN ('OPEN', 'CLOSED', 'PENDING', 'UNDER_REVIEW'))")
print("Constraint added: valid_status")


In [1]:
displayHTML("""
<details style="margin: 10px 0;">
  <summary style="cursor: pointer; color: #1B3A4B; font-weight: bold; padding: 8px 0;">Click to reveal hints</summary>
  <div style="background: #F9F7F4; padding: 12px 16px; border-radius: 8px; margin-top: 8px; border-left: 4px solid #F9A825;">
    <ul style="margin: 8px 0; padding-left: 20px;"><li style="margin: 6px 0; color: #1B1F24;">Answer: <code>DESCRIBE HISTORY</code></li></ul>
  </div>
</details>
""")

Click to reveal hints 
 
 Answer: DESCRIBE HISTORY

In [ ]:
# Demo 6.2 - Try to insert a claim with negative amount
bad_schema = StructType([
    StructField("claim_id", StringType()),
    StructField("policy_id", StringType()),
    StructField("axa_entity_id", StringType()),
    StructField("claim_date", StringType()),
    StructField("claim_amount", DoubleType()),
    StructField("claim_type", StringType()),
    StructField("claim_status", StringType()),
    StructField("adjuster_id", StringType()),
    StructField("priority", StringType())
])

bad_claim = spark.createDataFrame([
    ("CLM-BAD-001", "POL-000", "ENT_001", "2025-03-01", -5000.0, "AUTO", "OPEN", "ADJ-099", None)
], bad_schema)

try:
    bad_claim.write.format("delta").mode("append").saveAsTable(DEMO_TABLE)
    print("Write succeeded -- constraint did NOT work!")
except Exception as e:
    print("Constraint VIOLATION detected (as expected).")
    print(f"Error: {str(e)[:300]}")


In [ ]:
# Demo 6.3 - See all table properties, including constraints
display(spark.sql(f"""
SHOW TBLPROPERTIES {DEMO_TABLE}
"""))


In [1]:
displayHTML("""
<div style="border-left: 4px solid #00A972; padding: 10px 15px; margin: 10px 0; background-color: #F1F8E9; color: #1B1F24;">
<strong>Key Takeaway:</strong> CHECK constraints enforce business rules (positive amounts, valid statuses) at the table level. Any write that violates a constraint is rejected entirely. Use <code>SHOW TBLPROPERTIES</code> to verify which constraints are active.
</div>
""")

Key Takeaway: CHECK constraints enforce business rules (positive amounts, valid statuses) at the table level. Any write that violates a constraint is rejected entirely. Use SHOW TBLPROPERTIES to verify which constraints are active.

---
## What You Learned

In [1]:
displayHTML("""
<div style="background: #F9F7F4; padding: 20px; border-radius: 8px; border: 1px solid #dee2e6; margin: 10px 0;">

<table style="width:100%; border-collapse: collapse; font-size: 13px; color: #1B1F24;">
<tr style="background-color: #1B3A4B; color: #FFFFFF;">
  <th style="padding: 10px; text-align: left;">Concept</th>
  <th style="padding: 10px; text-align: left;">What It Solves</th>
  <th style="padding: 10px; text-align: left;">Key Command</th>
</tr>
<tr style="background-color: #FFFFFF;">
  <td style="padding: 8px; border-bottom: 1px solid #dee2e6; color: #1B1F24;"><strong>Delta Format</strong></td>
  <td style="padding: 8px; border-bottom: 1px solid #dee2e6; color: #1B1F24;">Raw files have no guarantees</td>
  <td style="padding: 8px; border-bottom: 1px solid #dee2e6; color: #1B1F24;"><code>.format("delta").saveAsTable()</code></td>
</tr>
<tr style="background-color: #F9F7F4;">
  <td style="padding: 8px; border-bottom: 1px solid #dee2e6; color: #1B1F24;"><strong>ACID / MERGE</strong></td>
  <td style="padding: 8px; border-bottom: 1px solid #dee2e6; color: #1B1F24;">Partial writes, concurrent corruption</td>
  <td style="padding: 8px; border-bottom: 1px solid #dee2e6; color: #1B1F24;"><code>MERGE INTO ... USING ... ON ...</code></td>
</tr>
<tr style="background-color: #FFFFFF;">
  <td style="padding: 8px; border-bottom: 1px solid #dee2e6; color: #1B1F24;"><strong>Schema Enforcement</strong></td>
  <td style="padding: 8px; border-bottom: 1px solid #dee2e6; color: #1B1F24;">Silent data corruption from schema drift</td>
  <td style="padding: 8px; border-bottom: 1px solid #dee2e6; color: #1B1F24;">Automatic (write is rejected)</td>
</tr>
<tr style="background-color: #F9F7F4;">
  <td style="padding: 8px; border-bottom: 1px solid #dee2e6; color: #1B1F24;"><strong>Schema Evolution</strong></td>
  <td style="padding: 8px; border-bottom: 1px solid #dee2e6; color: #1B1F24;">Need to add new columns</td>
  <td style="padding: 8px; border-bottom: 1px solid #dee2e6; color: #1B1F24;"><code>.option("mergeSchema", "true")</code></td>
</tr>
<tr style="background-color: #FFFFFF;">
  <td style="padding: 8px; border-bottom: 1px solid #dee2e6; color: #1B1F24;"><strong>Time Travel</strong></td>
  <td style="padding: 8px; border-bottom: 1px solid #dee2e6; color: #1B1F24;">Accidental deletes, auditing</td>
  <td style="padding: 8px; border-bottom: 1px solid #dee2e6; color: #1B1F24;"><code>VERSION AS OF n</code> / <code>RESTORE TABLE</code></td>
</tr>
<tr style="background-color: #F9F7F4;">
  <td style="padding: 8px; border-bottom: 1px solid #dee2e6; color: #1B1F24;"><strong>Delta Log</strong></td>
  <td style="padding: 8px; border-bottom: 1px solid #dee2e6; color: #1B1F24;">Debugging, understanding operations</td>
  <td style="padding: 8px; border-bottom: 1px solid #dee2e6; color: #1B1F24;"><code>_delta_log/*.json</code></td>
</tr>
<tr style="background-color: #FFFFFF;">
  <td style="padding: 8px; color: #1B1F24;"><strong>CHECK Constraints</strong></td>
  <td style="padding: 8px; color: #1B1F24;">Bad data passing schema checks</td>
  <td style="padding: 8px; color: #1B1F24;"><code>ALTER TABLE ADD CONSTRAINT ... CHECK (...)</code></td>
</tr>
</table>

</div>
""")

Concept,What It Solves,Key Command
Delta Format,Raw files have no guarantees,".format(""delta"").saveAsTable()"
ACID / MERGE,"Partial writes, concurrent corruption",MERGE INTO ... USING ... ON ...
Schema Enforcement,Silent data corruption from schema drift,Automatic (write is rejected)
Schema Evolution,Need to add new columns,".option(""mergeSchema"", ""true"")"
Time Travel,"Accidental deletes, auditing",VERSION AS OF n / RESTORE TABLE
Delta Log,"Debugging, understanding operations",_delta_log/*.json
CHECK Constraints,Bad data passing schema checks,ALTER TABLE ADD CONSTRAINT ... CHECK (...)


### Next Steps - Day 2

In Day 2, you will build on these Delta Lake foundations:
- **Medallion Architecture**: Organize data into Bronze, Silver, and Gold layers
- **Delta Live Tables (DLT)**: Declarative ETL pipelines with automatic dependency management
- **OPTIMIZE and Z-ORDER**: Performance tuning for Delta tables at scale
- **VACUUM**: Clean up old files while respecting time travel retention

---
## Cleanup

In [ ]:
# Cleanup (optional)
# spark.sql(f"DROP TABLE IF EXISTS {DEMO_TABLE}")
# spark.sql("DROP TABLE IF EXISTS {DATABASE}.claims_raw")
# dbutils.fs.rm(DELTA_EXPLORATION_PATH, recurse=True)
# print("Lab 03 cleanup complete.")